In [1]:
import cv2
import mediapipe as mp

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

print("FaceMesh ready!")

FaceMesh ready!


In [8]:
cap = cv2.VideoCapture(0)

mp_face_mesh = mp.solutions.face_mesh

# Custom styles
mesh_style = mp_drawing.DrawingSpec(color=(80, 220, 255), thickness=1, circle_radius=0)      # cyan mesh lines
contour_style = mp_drawing.DrawingSpec(color=(0, 255, 120), thickness=2, circle_radius=0)     # green face contour
iris_style = mp_drawing.DrawingSpec(color=(255, 0, 200), thickness=2, circle_radius=0)        # magenta iris

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            break

        results = face_mesh.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                # Fine tessellation mesh
                mp_drawing.draw_landmarks(
                    image=image,
                    landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_TESSELATION,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=mesh_style
                )
                # Face contours (eyes, lips, eyebrows, jawline)
                mp_drawing.draw_landmarks(
                    image=image,
                    landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_CONTOURS,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=contour_style
                )
                # Iris tracking
                mp_drawing.draw_landmarks(
                    image=image,
                    landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_IRISES,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=iris_style
                )

        cv2.imshow("FaceMesh", cv2.flip(image, 1))
        if cv2.waitKey(100) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()